In [1]:
import pandas as pd
import cv2
import numpy as np
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from src.blank_detect import is_blank

In [2]:
BASE_DIR = Path.cwd()
CSV_PATH = BASE_DIR / "data" / "output" / "dataset" / "labels.csv"
IMG_DIR = BASE_DIR / "data" / "output" / "dataset" / "images"

In [3]:
df = pd.read_csv(CSV_PATH)

images = []
labels = df["is_blank"].values

In [4]:
for filename in df["filename"]:
    img_path = str(IMG_DIR / filename)
    
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE) 
    
    if img is None:
        print(f"Warning: Could not read {img_path}")
        continue
        
    images.append(img)

In [5]:
X_data = np.array(images)
y_data = labels

print(f"Images array shape: {X_data.shape}")
print(f"Labels array shape: {y_data.shape}")

Images array shape: (500, 500, 500)
Labels array shape: (500,)


In [21]:
df['detected_blank'] = [
    int(is_blank(cv2.imread(str(IMG_DIR / f)), lower_threshold=0.005)) 
    for f in df['filename']
]

df['modified_y'] = (~df['content_type'].isin(['mixed', 'text_only'])).astype(int)

In [22]:
df.head(20)

,filename,is_blank,content_type,detected_blank,modified_y
0,img_0000.png,1,blank,1,1
1,img_0001.png,0,noise_only,1,1
2,img_0002.png,1,blank,1,1
3,img_0003.png,1,blank,1,1
4,img_0004.png,0,noise_only,1,1
5,img_0005.png,1,blank,1,1
6,img_0006.png,0,noise_only,1,1
7,img_0007.png,1,blank,1,1
8,img_0008.png,0,dots_only,1,1
9,img_0009.png,0,noise_only,1,1


In [23]:
y_true = df['modified_y']
y_pred = df['detected_blank']

print("--- Classification Report ---")
print(classification_report(y_true, y_pred, target_names=['Not Blank (0)', 'Blank (1)']))

--- Classification Report ---
               precision    recall  f1-score   support

Not Blank (0)       0.71      0.86      0.78       194
    Blank (1)       0.90      0.78      0.83       306

     accuracy                           0.81       500
    macro avg       0.80      0.82      0.81       500
 weighted avg       0.83      0.81      0.81       500

